# Generazione Dataset avanzato usando Gemma

# Descrizione del problema

All'interno di questo Jupyter Notebook è presente la pipeline anticipata nel punto 6 del Notebook 1: Generazione Dataset Elementare.

Nel primo notebook è stato infatti generato un dataset sintetico elementare. Le sue caratteristiche portano in overfitting gli algoritmi di Machine Learning analizzati nella pipeline del punto 03. Avevo inizialmente optato per la generazione di un dataset utilizzando Gemini con un normalissimo prompt, ma il risultato era troppo elementare e soprattutto poco accademico. Ho deciso dunque di adottare una soluzione decisamente più moderna e soprattutto scalabile a qualsiasi contesto.

La soluzione adottata è stata utilizzare Gemma4, una famiglia di modelli linguistici di grandi dimensioni (Large Language Model) sviluppata da Google che ho configurato ed eseguito localmente tramite la piattaforma Ollama. Questa architettura ha permesso di non limitarmi nella scrittura di un prompt per la generazione di n ticket, ma strutturare una funzione che, passati parametri come categoria, priorità, colloquialità, ecc... creasse in **run time** dei prompt casuali per la generazione di **singoli ticket**, per simulare le richieste dei singoli utenti, nella vita reale tutti diversi a modo loro in quanto esseri umani.

### Come eseguire il codice di questo Jupyter Notebook

__Questo Jupyter Notebook contiene già del codice eseguito a scopo dimostrativo, tuttavia a seguire è disponibile una breve guida passo-passo qualora si desiderasse configurare l'ambiente ed eseguirlo nuovamente__

*Questa sezione è effettivamente la più complessa da replicare autonomamente, in quanto è richiesta l'installazione di un'applicazione e di un modello di mediamente una decina di GB. Potrebbe inoltre essere richiesta una maggiore potenza di calcolo. Nonostante questo, esso è comunque perfettamente replicabile per intero.*

Per eseguire correttamente questo Notebook e avviare la generazione del dataset tramite modello LLM in locale, occorre seguire questi passaggi preliminari:
- Ollama deve essere installato direttamente sul sistema operativo per poter ospitare il modello in locale. Qualsiasi sia il sistema operativo, basta scaricare e installare l'applicazione ufficiale dal sito [ollama.com](https://ollama.com).
- Ad installazione finita, dovrà essere scaricato ed installato il modello. Da powershell, basta runnare il comando: *ollama run gemma4:e2b* (7.2 GB richiesti)
- A fine procedura, qualora si desiderasse rimuovere il modello, basterà utilizzare il comando: *ollama rm gemma4:e2b*

_Installazione di librerie utilizzate nel progetto:_ 

In [ ]:
%pip install pandas ollama pydantic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Sviluppo del codice

Import delle librerie:

In [1]:
import json
import pandas as pd
import random
from ollama import Client
from pydantic import BaseModel, Field

Struttura base dei parametri di generazione. Viene creata una classe *TicketSintetico* ereditando la classe base di pydantic *BaseModel*. Questa operazione permette di avere tutto il controllo necessario sull'output generato dal modello di LLM, qualsiasi modello venga utilizzato. In particolare, ciò che restituirà è un dizionario formato da un titolo e da un corpo, entrambi con una descrizione della struttura. Questo inoltre evita allucinazioni e permette l'utilizzo di una formattazione standard.

In [2]:
class TicketSintetico(BaseModel):
    title: str = Field(
        description="Un titolo breve, specifico e realistico."
    )
    body: str = Field(
        description=(
            "Il corpo del ticket. Deve contenere una descrizione breve (1-2 frasi) del problema reale. "
            "NON usare placeholder, parentesi quadre, simboli di markup o istruzioni di compilazione (es. '[Descrivere qui...]'). "
            "Non è obbligatorio inserire nominativi di applicazioni/servizi ma, se lo fai, utilizza nomi verosimili (non usare ad esempio modulo X o endpoint x)"
            "Il testo deve essere testo pronto, discorsivo e già compilato, scritto dal punto di vista dell'utente o del dipendente che sta riscontrando il problema in prima persona"
        )
    )

Funzione della generazione di un singolo ticket. Qui viene formattato il prompt della richiesta da fare al modello per la creazione di un determinato ticket basandoci su dei parametri:
- categoria: la categoria del ticket
- priorità: la priorità del problema
- esperienza: il livello di esperienza nel settore specifico della categoria da parte dell'utente
- colloquialità: il tono da utilizzare per la stesura del ticket
- nome_utente: informazione extra da utilizzare in maniera facoltativa che rappresenta se voler aggiungere il nome dell'utente durante la stesura del ticket
- typo: booleano che specifica se richiedere l'inserimento di un typo o meno

L'output della funzione sarà una tupla, contenente body e title del ticket.

In [3]:
client = Client()
def genera_ticket_con_gemma(categoria, priorita,esperienza,colloquialità,incipit,titolo,disturbo=False,typo=False):
    prompt_utente = f"""Genera un ticket di assistenza aziendale in lingua italiana con queste caratteristiche:
    - Categoria del ticket: {categoria}. Questo rappresenta l'argomento principale di cui deve parlare il ticket.
    - Priorità del problema: {priorita}.
    - Livello di esperienza dell'utente: {esperienza}.
    - Livello di colloquialità/tono: {colloquialità}
  
    STILE DI SCRITTURA:
    1. Sii estremamente sintetico. Il testo deve essere composto da massimo 2-3 frasi corte.
    2. Scrivi in prima persona. 
    3. Non usare MAI caratteri speciali di formattazione come asterischi (*), corsivi o grassetti.
    4. Evita deliri o flussi di coscienza lunghi: vai dritto al problema reale.
    5. La forma di saluto iniziale deve essere esattamente "{incipit}" (se non è vuota) o una sua variante naturale.
    6. Usa termini tecnici, acronimi o parole chiave specifiche in base alla categoria del ticket ({categoria}) e all'esperienza ({esperienza}).
    7. Il titolo deve simulare l'oggetto scritto da un utente reale (breve, sbrigativo o imperfetto). Inizia il titolo UTILIZZANDO OBBLIGATORIAMENTE questa esatta espressione iniziale: "{titolo}". Continua poi la frase completando l'oggetto in modo che si colleghi perfettamente alla categoria ({categoria}) e soprattutto alla priorità ({priorita}).
    8. Rispetta rigorosamente la coerenza: se la priorità è Bassa, non inserire scadenze imminenti o blocchi operativi nel titolo o nel corpo
    """
    if disturbo:
        prompt_utente+="""\n - VINCOLO DI CONTESTO DINAMICO (OBBLIGATORIO):
                        - {disturbo}. Integra questo dettaglio nel testo in modo naturale, usandolo come elemento di sfondo, senza farlo diventare il problema principale."""

    if typo:
        prompt_utente += "\n- Inserisci dove rendi opportuno un typo per rendere il ticket più human like."

    try:
        response = client.chat(
            model='gemma4:e2b', 
            messages=[{'role': 'user', 'content': prompt_utente}],
            format={
                'type': 'object',
                'properties': {
                    'title': {'type': 'string'},
                    'body': {'type': 'string'}
                },
                'required': ['title', 'body']
            },
            options={
                'keep_alive': -1,  # -1 dice a Ollama: "Non scaricare MAI il modello finché non si chiude lo script"
                'temperature': 0.4
            }
        )
        dati_ticket = json.loads(response['message']['content'].strip())
        return dati_ticket['title'], dati_ticket['body']
    except Exception as e:
        print(f"Errore durante la generazione locale: {e}")
        return None, None


Definizione dei Pool di Input:
- Delle semplicissime liste contenenti dei pool di categorie, priorità, livelli di esperienza, ecc... che verranno estratti casualmente all'interno del ciclo. Il loro compito è quello di fornire un incipit di partenza a Gemma, così da indirizzarla di volta in volta nella crezione del ticket (operazione necessaria dato che inizialmente il modello creava sempre ticket simili)

In [4]:
categorie_disponibili = ["Tecnico", "Amministrazione", "Commerciale"]
priorita_disponibili = ["Bassa", "Media", "Alta"]
livelli_esperienza = [
    "Principiante", "Intermedio", "Esperto/Professionale", "Utente occasionale",
    "Primo utente", "Avanzato", "Primo utilizzo", "Utente che si crede esperto ma non lo è",
    "Tecnofobo / Anziano", "Nativo digitale", "Sviluppatore / Tecnico interno",     
    "Amministratore di sistema (Admin)","Utente pigro","Cliente VIP / Direzione",            
    "Utente metodico/Preciso","Ex-utente di un software concorrente"
]
livelli_colloquialita = [
    "Formale e distaccato", "Molto informale e colloquiale", "Frustrato/Arrabbiato",
    "Confuso", "Dettagliato", "Frettoloso", "Sarcastico", "Simpaticone", "Ansioso / Nel panico",
    "Disperato / Rassegnato", "Logorroico / Prolisso", "Telegrafico / Minimale","Passivo-aggressivo",
    "Gentile ed empatico", "Disorganizzato / Sconnesso", "Diffidente / Sospettoso"
]
pool_incipit_urgenti = [
    "Così non posso lavorare! ", 
    "Non so più che fare... ", 
    "Da stamattina sono completamente bloccato a ",
    "Ho un'urgenza assoluta: ",
    "Scrivo perché l'attività è interrotta: ",
    "Ho bisogno di risolvere subito ",
    "Siamo completamente fermi su ",
    "Blocco totale della produzione su ",
    "Disastro, non funziona più nulla dopo ",
    "Soluzione immediata richiesta per ",
    "È prioritario sbloccare ",
    "Impossibile procedere con il lavoro a causa di ",
    "Mi serve supporto ora, sono fermo su ",
    "Situazione critica su ",
    "Non si può andare avanti così! "
]

pool_incipit_normali = [
    "",
    "Ciao, ", 
    "Buongiorno. ", 
    "Buongiorno! ",
    "Eccomi di nuovo, ", 
    "Salve, ", 
    "Buongiorno, ", 
    "Scusate il disturbo, ", 
    "Ho un problema: ", 
    "Vi scrivo perché ", 
    "Creo questo ticket per ", 
    "Da stamattina sto provando a ", 
    "Dopo l'ultimo aggiornamento, ", 
    "Ho già provato a riavviare ", 
    "Potete dare un occhiata a ", 
    "Ho già provato a riavviare, ma ", 
    "Gentile supporto, vi contatto per ", 
    "Vorrei segnalare ", 
    "Ho necessità di segnalare ",
    "Avrei bisogno di un controllo su ",
    "Vi contatto per una anomalia riscontrata su ",
    "Riscontro un errore nel sistema quando provo a ",
    "Non riesco a completare la procedura di ",
    "Sembra esserci un bug su ",
    "Eccoci di nuovo... "
]

pool_incipit_bassa = [
    "",
    "Quando avete tempo, ",
    "Senza alcuna fretta, ",
    "Giusto per informazione, ",
    "Vi scrivo per una questione non urgente: ",
    "Avrei una curiosità su ",
    "Segnalo una piccolissima anomalia: ",
    "Un piccolo dubbio su ",
    "Nulla di bloccante, ma volevo segnalare che ",
    "Per il futuro sarebbe utile capire se ",
    "Solo una domanda di routine: ",
    "Quando capita, potreste dare un'occhiata a ",
    "Solo una richiesta di info su ",
    "Piccolo chiarimento per quando siete liberi: ",
    "Volevo solo lasciarvi un feedback riguardo a "
]

pool_titolo_urgenti = [
    "",
    "Urgente: non va ", 
    "Richiesta urgente: ", 
    "Assistenza immediata per ",
    "BLOCCO TOTALE: ",
    "CRITICO - Malfunzionamento su ",
    "EMERGENZA: ",
    "Interruzione servizio su "
]

pool_titolo_normali = [
    "",
    "Problema con ", 
    "Aiuto: ", 
    "Errore ", 
    "Bloccato su ", 
    "Info per ", 
    "Non va ", 
    "Chiarimento su ", 
    "Mancato funzionamento ", 
    "Procedura per ", 
    "Richiesta chiarimenti ",
    "Segnalazione errore su ",
    "Anomalia riscontrata in ",
    "Richiesta di supporto per "
]

pool_titolo_bassa = [
    "",
    "Richiesta info (non urgente): ",
    "Segnalazione minore su ",
    "Domanda generica: ",
    "Feedback su ",
    "Piccolo dubbio su ",
    "Suggerimento / Miglioria per ",
    "Info di routine per ",
    "Nota di aggiornamento su ",
    "Curiosità tecnica riguardo a "
]
reparti_aziendali = [
    "Reparto Logistica", "Ufficio Risorse Umane", "Ufficio Acquisti", 
    "Produzione", "Marketing", "Sede di Roma", "Filiale Estera", 
    "Reparto tecnico", "Cliente",
    "Ufficio Contabilità", "Controllo Qualità", "Ufficio Legale", 
    "Reparto Vendite", "Sede di Milano", "Assistenza Clienti", 
    "Ufficio Personale", "Management", "Direzione Generale", 
    "Fornitore Esterno"
]

strumenti_generici = [
    "gestionale interno", "portale dipendenti", "piattaforma cloud", 
    "software di rete", "sistema centrale", "CRM", "applicazione", 
    "servizio cloud",
    "database aziendale", "server centrale", "portale fornitori", 
    "applicazione di fatturazione", "pannello di controllo", 
    "sistema di login", "software di contabilità", "piattaforma HR", 
    "modulo richieste", "archivio digitale"
]

Semplicissimo ciclo for responsabile della creazione di *da_fare* ticket. 
- All'inizio è possibile notare la presenza di vari random.choices pesati per creare una logica realistica dei dati che verranno realizzati;
- *titolo, corpo = genera_ticket_con_gemma(categoria, priorità, esperienza, colloquialità,incipit,titolo,disturbo, attiva_typo)*, chiama la funzione precedentemente descritta e conserva all'interno delle variabili *titolo* e *corpo* il risultato;
- Viene creato un dizionario e convertito in un dataframe, inserito infine all'interno del file .csv.

In [ ]:
import pandas as pd
import random

percorso_csv = "dataset/ticket_gemma4e2b.csv"

da_fare = 5

#controllo se è il primo inserimento o meno
try:
    df_esistente = pd.read_csv(percorso_csv)
    
    if not df_esistente.empty and 'id' in df_esistente.columns:
        ultimo_id_str = df_esistente['id'].iloc[-1]
        id_di_partenza = int(ultimo_id_str.split("-")[1]) + 1
    else:
        id_di_partenza = 0
        
    scrivi_header = False
    print(f"File CSV rilevato con Pandas. Riprendo dall'ID: TK-{id_di_partenza}")

except (FileNotFoundError, pd.errors.EmptyDataError):
    #parto da zero se il file non è stato trovato o è vuoto
    id_di_partenza = 0
    scrivi_header = True
    print("File CSV non trovato o vuoto. Creazione nuovo dataset da TK-0.")

print("-" * 50)

for i in range(0, da_fare):
    id_corrente_numerico = id_di_partenza + i
    stringa_id = f"TK-{id_corrente_numerico}"
    
    categoria = random.choice(categorie_disponibili)
    priorità = random.choice(priorita_disponibili)
    esperienza = random.choice(livelli_esperienza)
    colloquialità = random.choice(livelli_colloquialita)
    attiva_typo = random.choices([True, False], weights=[20, 80])[0]

    #si sceglie incipit in base alla priorità. si usa la meccanica dei pesi per creare rumore
    if priorità == "Alta":
        pool_scelta_titolo = random.choices([pool_titolo_urgenti, pool_titolo_normali], weights=[70, 30])[0]
        titolo = random.choice(pool_scelta_titolo)
        
        pool_scelta_incipit = random.choices([pool_incipit_urgenti, pool_incipit_normali], weights=[70, 30])[0]
        incipit = random.choice(pool_scelta_incipit)

    elif priorità == "Media":
        pool_scelta_titolo = random.choices([pool_titolo_urgenti, pool_titolo_normali, pool_titolo_bassa], weights=[20, 60, 20])[0]
        titolo = random.choice(pool_scelta_titolo)
        
        pool_scelta_incipit = random.choices([pool_incipit_urgenti, pool_incipit_normali, pool_incipit_bassa], weights=[20, 60, 20])[0]
        incipit = random.choice(pool_scelta_incipit)

    elif priorità == "Bassa":
        pool_scelta_titolo = random.choices([pool_titolo_normali, pool_titolo_bassa], weights=[30, 70])[0]
        titolo = random.choice(pool_scelta_titolo)
        
        pool_scelta_incipit = random.choices([pool_incipit_normali, pool_incipit_bassa], weights=[30, 70])[0]
        incipit = random.choice(pool_scelta_incipit)    

    disturbo = ""
    if random.randint(0,2)==1:
        if categoria == "Tecnico":
            disturbo = f"L'utente specifica di lavorare presso: {random.choice(reparti_aziendali)}"
        elif categoria == "Amministrazione":
            disturbo = f"L'operazione contabile avviene tramite lo strumento: {random.choice(strumenti_generici)}"
        else:  # Commerciale
            disturbo = f"L'utente menziona che l'offerta serve per il budget del: {random.choice(reparti_aziendali)} o per lo strumento  {random.choice(strumenti_generici)}"

    print(f"Generazione in corso... [{categoria} - {priorità} - {esperienza} - {colloquialità} - {attiva_typo}] ({i+1}/{da_fare})")
            
    titolo, corpo = genera_ticket_con_gemma(categoria, priorità, esperienza, colloquialità,incipit,titolo,disturbo, attiva_typo)
    
    if titolo and corpo:
        singolo_ticket = {
            "id": [stringa_id],
            "title": [titolo],
            "body": [corpo],
            "category": [categoria],
            "priority": [priorità]
        }
        df_singolo = pd.DataFrame(singolo_ticket)
        
        #se è il primo ticket assoluto creato da zero, scrive l'header e apre in 'w'
        if id_corrente_numerico == 0 and scrivi_header:
            df_singolo.to_csv(percorso_csv, index=False, mode='w', header=True)
            scrivi_header = False 
        else:
            df_singolo.to_csv(percorso_csv, index=False, mode='a', header=False)
            
        print(f"Ticket {stringa_id} Creato e SALVATO nel CSV!")
        print(f"Titolo: {titolo}")
        print(f"Corpo:  {corpo[:80]}...") 
        print("-" * 50)
        print()
    else:
        print("Generazione fallita. Salto al successivo.")
        print("-" * 50)

print(f"\nGenerazione conclusa con successo!")
print(f"Trovi tutti i dati aggiornati in tempo reale qui: '{percorso_csv}'.")


File CSV rilevato con Pandas. Riprendo dall'ID: TK-495
--------------------------------------------------
Generazione in corso... [Amministrazione - Bassa - Primo utente - Logorroico / Prolisso - True] (1/5)
Ticket TK-495 Creato e SALVATO nel CSV!
Titolo: Problemi con la gestione dati amministrativi
Corpo:  Per il futuro sarebbe utile capire se posso avere assistenza sulla corretta conf...
--------------------------------------------------

Generazione in corso... [Tecnico - Media - Primo utente - Sarcastico - True] (2/5)
Ticket TK-496 Creato e SALVATO nel CSV!
Titolo: Non va la connessione
Corpo:  Non riesco a completare la procedura di riconnettere il modem. Il router non si ...
--------------------------------------------------

Generazione in corso... [Commerciale - Bassa - Ex-utente di un software concorrente - Frettoloso - False] (3/5)
Ticket TK-497 Creato e SALVATO nel CSV!
Titolo: Chiarimento su licenza software concorrente
Corpo:  Vi scrivo per una questione non urgente: Ho bi

*I ticket sono stati creati eseguendo il codice in modalità batch, variando ogni volta la struttura del prompt, inserendo o rimuovendo dettagli, con lo scopo di rendere il dataset più eterogeneo. I prompt e gli incipit presenti in questo Notebook sono quelli che hanno dato dei risultati più bilanciati.*

Per tale motivo, questa cella esegue un piccolo shuffling dei record (mantenendo l'ordine degli id) con lo scopo di neutralizzare eventuali bias legati alla sequenzialità delle varie fasi. 

In [ ]:
import pandas as pd
df = pd.read_csv("dataset/ticket_gemma4e2b.csv")
#isolo per conservare la colonna degli id
ids = df['id'].copy()
#mischio casualmente tutte le altre colonne
df_features_shuffled = df[['title', 'body', 'category', 'priority']].sample(frac=1, random_state=42).reset_index(drop=True)
df_omogeneo = pd.concat([ids, df_features_shuffled], axis=1)
df_omogeneo.to_csv(percorso_csv, index=False, mode='w', header=True)

# Conclusioni

- Inizialmente la scelta era ricaduta sulla versione standard e completa del modello Gemma 4. Sebbene l'accuratezza e la diversificazione dei testi fossero ottimali, ho notato una complessità computazionale non da ignorare. Nonostante il mio computer sia molto performante, il tempo di generazione medio di un ticket era di 2 al minuto. Il che significa circa 4 ore di elaborazione per generare un dataset di 500 ticket. *Inoltre, la pipeline ha riportato dei dati veramente deludenti, complice la complessità dei ticket.*. Tutto questo mi ha portato alla ricerca di alternative più leggere e sostenibili localmente.

- Ho pertanto sperimentato l'introduzione di varianti ottimizzate e quantizzate a parametri ridotti, nello specifico le release da 2 miliardi (Gemma4:e2b) e 4 miliardi di parametri (Gemma4:e4b). Queste architetture hanno garantito un'esecuzione  più fulminea e un'impronta di memoria minima sul sistema, al costo di una forte "pigrizia semantica"; si sono presentati fenomeni di ripetitività strutturale, generando comunque risultati migliori dei ticket generati nel Jupyter 01.

- Pertanto, il dataset che poi si è rivelato quello con i risultati migliori anche nella pipeline ML è quello generato con Gemma4:e2b. Il tempo di generazione medio di un ticket è sceso a 5 al minuto, dunque meno di 2 ore di elaborazione per generare un dataset di 500 ticket.

**NOTA: la macchina utilizzata per l'elaborazione dispone di un processore 12th Gen Intel(R) Core(TM) i5-12450H (2.00 GHz), 16.0 GB di RAM installata e scheda grafica NVIDIA GeForce RTX 4050 Laptop GPU (6 GB).**

Diamo un'occhiata alle varie versioni di dataset sintetici generati da Gemma4, al fine di validarne la qualità. All'interno della pipeline di Machine Learning sarà necessario svolgere un'EDA avanzata dei dati.

- Ticket Gemma4 unbalanced: dataset generato con la versione base di Gemma4, unbalanced perché il numero di ticket a priorità alta è volontariamente inferiore rispetto agli altri, con lo scopo di creare maggiore veridicità ai dati, che nella vita reale possono essere sbilanciati;
- Ticket Gemma4 balanced: stessa cosa della versione sopra, ma il numero di ticket per categoria e priorità è bilanciato;
- Ticket Gemma4e2b: stessa cosa, ma dataset realizzato per intero con la versione più leggera di Gemma.

In [5]:
import pandas as pd
df_anteprima = pd.read_csv("dataset/ticket_gemma4_unbalanced.csv")
print("*** TICKET GEMMA4 unbalanced ***")
df_anteprima.sample(10)

*** TICKET GEMMA4 unbalanced ***


,id,title,body,category,priority
235,TK-235,Assistenza immediata per verifica parametri co...,Vi scrivo perché ho notato un lieve scostament...,Amministrazione,Bassa
174,TK-174,Mancato funzionamento dell'accesso al portale HR,"Salve, ho notato che non riesco ad accedere al...",Amministrazione,Bassa
291,TK-291,Bloccato su configurazione API per integrazion...,Ho bisogno di fare una call riguardo un proble...,Tecnico,Alta
225,TK-225,Aiuto: Non riesco a capire i costi di questa p...,"Gentile supporto, vi contatto perché non capis...",Commerciale,Media
285,TK-285,Chiarimento su parametri di connettività VPN r...,Da stamattina sto provando a stabilire la conn...,Tecnico,Bassa
266,TK-266,Richiesta chiarimenti sulla procedura di acces...,Buongiorno. Sto riscontrando difficoltà nell'a...,Tecnico,Bassa
495,TK-495,Aiuto: Non trovo l'ultima versione del modulo ...,"Ciao, ho bisogno di recuperare la ultima versi...",Amministrazione,Media
153,TK-153,Non funziona la stampa del report mensile,"Ciao, non riesco proprio a stampare questo rep...",Tecnico,Bassa
249,TK-249,Errore nell'accesso al portale clienti dopo ag...,Da stamattina sto provando a connettermi all'a...,Commerciale,Bassa
118,TK-118,Non funziona la generazione di report trimestrali,"Buongiorno, ho notato che i dati per i report ...",Commerciale,Bassa


In [6]:
import pandas as pd
df_anteprima = pd.read_csv("dataset/ticket_gemma4_balanced.csv")
print("*** TICKET GEMMA4 balanced ***")
df_anteprima.sample(10)

*** TICKET GEMMA4 balanced ***


,id,title,body,category,priority
20,TK-20,Richiesta urgente: Problema di connettività AP...,Vi scrivo perché ho riscontrato un errore 503 ...,Tecnico,Media
277,TK-277,Info per la tariffa che mi è arrivata male,"Non volevo proprio dirlo, ma ho visto l'ultima...",Commerciale,Alta
22,TK-22,Non va l'accesso al sistema CRM dopo aggiornam...,Creo questo ticket per segnalare un problema d...,Tecnico,Bassa
417,TK-417,Assistenza immediata per blocco accesso al CRM...,"Gentile supporto, vi contatto perché non riesc...",Tecnico,Alta
471,TK-471,Errore quando provo ad accedere alla rete azie...,Ciao! Scusa se ti disturbo così. Non riesco pr...,Tecnico,Alta
315,TK-315,Chiarimento su parametri di connettività VPN r...,Da stamattina sto provando a stabilire la conn...,Tecnico,Bassa
324,TK-324,Aiuto: Non riesco a trovare i dati di fatturaz...,Non so più che fare. Ho bisogno di recuperare ...,Commerciale,Bassa
88,TK-88,Aiuto: Non riesco ad accedere alla sezione pre...,"Gentile supporto, vi contatto per un problema ...",Commerciale,Alta
365,TK-365,Aiuto: Non riesco ad accedere al sistema CRM,Scusate il disturbo. Ho problemi a loggarmi su...,Tecnico,Media
17,TK-17,Problema con l'accesso al database cliente dop...,Vi scrivo perché non riesco ad accedere al dat...,Tecnico,Alta


In [7]:
import pandas as pd
df_anteprima = pd.read_csv("dataset/ticket_gemma4e2b.csv")
print("*** TICKET GEMMA4:e2b ***")
df_anteprima.sample(10)

*** TICKET GEMMA4:e2b ***


,id,title,body,category,priority
66,TK-66,Chiarimento su problemi di configurazione soft...,Volevo solo lasciarvi un feedback riguardo a u...,Tecnico,Bassa
108,TK-108,EMERGENZA: Problemi con l'impostazione contabile,Scrivo perché l'attività è interrotta: Non so ...,Amministrazione,Alta
219,TK-219,Info per gestione utenti fallita,Il sistema di amministrazione non permette l'a...,Amministrazione,Media
415,TK-415,Errore gestione fatture alta priorità,Ho difficoltà a completare l'inserimento dei d...,Amministrazione,Alta
48,TK-48,Suggerimento / Miglioria per configurazione rete,Buongiorno! Ho riscontrato un piccolo inconven...,Tecnico,Bassa
409,TK-409,Segnalazione minore su gestione fatture arretrate,Ho notato un piccolo intoppo nella gestione de...,Amministrazione,Bassa
315,TK-315,Richiesta chiarimenti su errore sistema critico,È prioritario sbloccare la situazione. Ho un p...,Tecnico,Alta
325,TK-325,Errore gestione licenze software,Ho problemi con l'attivazione delle licenze do...,Commerciale,Media
292,TK-292,Problema con la gestione fatture,Ho difficoltà a visualizzare correttamente le ...,Amministrazione,Bassa
193,TK-193,Bloccato su connessione Wi-Fi,Vi contatto per una anomalia riscontrata su co...,Tecnico,Media


## Possibili Miglioramenti

Questo script mostra diversi possibili miglioramenti:
- Diversa strutturazione dei prompt: è possibile utilizzare diversi approcci di prompt engineering per avere risultati diversi e adatti alla situazione
- Diversi modelli di intelligenza artificiale: Gemma4 non è l'unico modello di intelligenza artificiale con il quale si può sfruttare questa pipeline in quanto sfrutta bene il paradigma della OOP. Per scegliere un altro modello da utilizzare, basta dare un'occhiata alla sezione model del sito web ufficiale di [ollama](https://ollama.com/search).

__Una EDA più accurata è disponibile all'interno del Jupyter 03.__